In [1]:
import numpy as np 
import pandas as pd 
import pathlib
import matplotlib.pyplot as pyplot
from neurodsp.spectral import compute_spectrum 
import os
from joblib import Parallel, delayed 
from tqdm import tqdm 
import scipy 
import pickle

from hnn_core import neymotin_2020_model, pick_connection
from hnn_core.optimization import Optimizer 
from hnn_core.optimization.objective_functions import _check_is_batch
from hnn_core.batch_simulate import  BatchSimulate
from scipy.interpolate import CubicSpline # a tool from the SciPy library for drawing a smooth curve

--No graphics will be displayed.


In [2]:
def get_eeg_data(subj_data_path): 
    channel_filter = ['F7', 'P8', 'T7', '02'] # electrodes we want to keep 

    eeg_dict = np.load(subj_data_path, allow_pickle=True) # load the saved file 
    channel_labels = eeg_dict['labels']
    sampling_freq = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    assert eeg_data.shape[0] == len(channel_filter) 

    return eeg_data

In [3]:
def add_bursty_opt_drives(net, tstop=200, n_prox=2, n_dist=1): 
    prox_cell_type = ["L5_pyramidal", "L5_basket", "L2_pyramidal", "L2_basket"]
    dist_cell_type = ["L5_pyramidal", "L2_pyramidal", "L2_basket"]
    default_range = {
        "burst_rate": (1, 100),
        "burst_std": (0, 500), 
        "ampa": (-5, 1), 
        "nmda": (-5, 1),
    }

    prox_weights = {cell_type: 0.0 for cell_type in prox_cell_type} # placeholder, add bursty drive needs a weight for every cell type it targets so you have to hand it something when you create the drive 
    dist_weights = {cell_type: 0.0 for cell_type in dist_cell_type}

    prox_delays = {
        "L2_basket": 0.1,
        "L2_pyramidal": 0.1,
        "L5_basket": 1.0, 
        "L5_pyramidal": 1.0}

    dist_delays = {
        "L2 basket": 0.1, 
        "L2_pyramidal": 0.1, 
        "L5_pyramidal": 0.1}
    
    constraints, initial_params = dict(), dict()
    # add proximal drives 
    for idx in range (n_prox): 
        name = f"bursty_prox{idx+1}"
        constraints[f"bursty_prox{idx+1}"]
        initial_params[f"{name}_burst_rate"] = np.random.uniform(*default_range["burst_rate"])

        constraints[f"{name}_burst_std"] = default_range["burst_std"]
        initial_params[f"{name}_burst_std"] = np.random.uniform(*default_range["burst_std"])

        for cell_type in prox_cell_type:
            constraints[f"{name}_{cell_type}_ampa"] = default_range["ampa"]
            initial_params[f"{name}_{cell_type}_ampa"] = np.random.uniform(
                *default_range["ampa"]
            )

            constraints[f"{name}_{cell_type}_nmda"] = default_range["nmda"]
            initial_params[f"{name}_{cell_type}_nmda"] = np.random.uniform(
                *default_range["nmda"]
            )

        net.add_bursty_drive(name,
                    tstart=0,
                    tstart_std=0,
                    tstop=None,
                    location='proximal',
                    burst_rate=1,
                    burst_std=0,
                    numspikes=2,
                    spike_isi=10,
                    n_drive_cells='n_cells',
                    cell_specific=True,
                    weights_ampa=prox_weights,
                    weights_nmda=prox_weights,
                    event_seed=2)                          # code here is in HNN notebook/public access?

    for idx in range(n_dist):
        name = f"bursty_dist{idx + 1}"
        constraints[f"{name}_burst_rate"] = default_range["burst_rate"]
        initial_params[f"{name}_burst_rate"] = np.random.uniform(*default_range["burst_rate"])

        constraints[f"{name}_burst_std"] = default_range["burst_std"]
        initial_params[f"{name}_burst_std"] = np.random.uniform(*default_range["burst_std"])


        for cell_type in prox_cell_type:
            constraints[f"{name}_{cell_type}_ampa"] = default_range["ampa"]
            initial_params[f"{name}_{cell_type}_ampa"] = np.random.uniform(
                *default_range["ampa"]
            )

            constraints[f"{name}_{cell_type}_nmda"] = default_range["nmda"]
            initial_params[f"{name}_{cell_type}_nmda"] = np.random.uniform(
                *default_range["nmda"]
            )

        net.add_bursty_drive(name,
                            tstart=0,
                            tstart_std=0,
                            tstop=None,
                            location='distal',
                            burst_rate=1,
                            burst_std=0,
                            numspikes=2,
                            spike_isi=10,
                            n_drive_cells='n_cells',
                            cell_specific=True,
                            weights_ampa=dist_weights,
                            weights_nmda=dist_weights,
                            event_seed=2)

    return constraints, initial_params

In [4]:
def set_params_bursty_opt_drives(net, param_values):             # goes through each drive in the network and writes the new burst rate and burst jitter into it 
    drive_names = list(net.external_drives.keys())
   
    for name in drive_names:
        target_cell_types = net.external_drives[name]["target_types"]  # look up which cell type this drive connects to
        net.external_drives[name]["dynamics"]["burst_rate"] = param_values[f"{name}_burst_rate"] # look up this drive's burst rate in param_values and store it in the drive
        net.external_drives[name]["dynamics"]["burst_std"] = param_values[f"{name}_burst_std"] # look up the drive's std deviation in param_values and store it the drive 

        for receptor in ["ampa", "nmda"]: 
            net.external_drives[name][f"weights_{receptor}"] = {
                ct: 0.0 for ct in target_cell_types                    # for each cell type this drive targets, a weight of 0.0 should be assigned, replace the drive's old record with the fresh one 
            }

    
        for cell_type in target_cell_types:
            for receptor in ["ampa", "nmda"]:      # for each cell type, loops over the two receptor types so every cell type is handled 2x 
                conn_idx = pick_connection(
                    net, src_gids=name, target_gids=cell_type, receptor=receptor      # searches the network's list of connections for the one that goes from the drive to cell type w/ receptor 
                )
                assert len(conn_idx) == 1         
                
                net.connectivity[conn_idx[0]]["nc_dict"]["A_weight"] = (          # the matched connection's position, connection's strength / number the simulation uses/builds parameter name 
                    10 ** param_values[f"{name}_{cell_type}_{receptor}"]
                )                                                              
                net.external_drives[name][f"weights_{receptor}"][cell_type] = (     
                    10 ** param_values[f"{name}_{cell_type}_{receptor}"]
                )

In [5]:
def _process_psd(dpl, target_freqs, dt):
    hnn_data = dpl[0].data['agg']
    burn_in = 250 # ms
    burn_in_samples = int(burn_in / dt)
    hnn_data = hnn_data[burn_in_samples:]

    fs_hnn = 1e3 / dt
    freqs_hnn, psd_hnn = compute_spectrum(hnn_data, fs=fs_hnn, method=method)
    cs = CubicSpline(freqs_hnn, psd_hnn)

    psd = cs(target_freqs)
    
    return psd

In [6]:
def _corr_psd(
    initial_net,
    initial_params,
    set_params,
    predicted_params,
    update_params,
    obj_values,
    tstop,
    obj_fun_kwargs,
):  

    is_batch = _check_is_batch(predicted_params)

    if is_batch:
        # params = update_params(initial_params, predicted_params)
        predicted_params = np.array(predicted_params).reshape(-1, len(initial_params))
        params_batch = {
            name: predicted_params[:, idx]
            for idx, name in enumerate(initial_params.keys())
        }

        # simulate dpl with predicted params
        new_net = initial_net.copy()

        def set_params_batch(a, b):
            set_params(b, a)  # need to fix this

        batch_simulation = BatchSimulate(
            net=new_net,
            set_params=set_params_batch,
            save_outputs=False,
            save_dpl=True,
            dt=obj_fun_kwargs.get("dt", 0.025),
            n_trials=obj_fun_kwargs.get("n_trials", 1),
            tstop=tstop,
            overwrite=False,
            clear_cache=False,
        )

        res = batch_simulation.run(
            params_batch,
            n_jobs=obj_fun_kwargs.get("n_jobs", 1),
            combinations=False,
            backend="loky",
            verbose=obj_fun_kwargs.get("verbose", True),
        )

        target_freqs, target_spectrum = obj_fun_kwargs["target"]
        dt = obj_fun_kwargs["dt"]

        psd_list = list()
        for batch_res in res["simulated_data"]:
            for data in batch_res:
                psd = _process_psd(data["dpl"], target_freqs, dt)
                psd_list.append(psd)

        obj = [np.corrcoef(psd, target_spectrum)[0,1] for psd in psd_list]

        obj = 1 - np.array(obj)

        obj_values.append(obj.tolist())

        print(f"Mean Loss: {np.mean(obj):.2f} ; Min Loss: {np.min(obj):.2f}")
        return obj

    else:
        print('Only batch simulations currently supported')

    print(f"Mean Loss: {np.mean(obj):.2f}; Min Loss: {np.min(obj):.2f}")
    obj_values.append(obj)

    return obj